# पाठ 02 - Microsoft एजेंट फ्रेमवर्क का अन्वेषण

**Microsoft एजेंट फ्रेमवर्क (MAF)** AI एजेंट बनाने के लिए एक एकीकृत फ्रेमवर्क है। यह चार मुख्य बिल्डिंग ब्लॉक्स के साथ एक साफ, संयोज्य वास्तुकला प्रदान करता है:

- **क्लाइंट** – AI मॉडल एंडपॉइंट से कनेक्ट होता है और संचार संभालता है
- **एजेंट** – क्लाइंट को निर्देशों और टूल परिभाषाओं के साथ लपेटता है
- **टूल्स** – एजेंट की क्षमताओं को कस्टम फ़ंक्शंस से बढ़ाते हैं जिन्हें मॉडल कॉल कर सकता है
- **सेशन** – बहु-चरण इंटरैक्शन के लिए बातचीत का इतिहास बनाए रखता है

इस पाठ में, हम इन अवधारणाओं का उपयोग करते हुए एक **ट्रैवल बुकिंग एजेंट** बनाएंगे जो गंतव्य उपलब्धता की जांच करता है।


## सेटअप


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## एजेंट फ्रेमवर्क आर्किटेक्चर को समझना

Microsoft एजेंट फ्रेमवर्क एक स्तरीय आर्किटेक्चर का पालन करता है:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **क्लाइंट** – एक `FoundryChatClient` Azure OpenAI डिप्लॉयमेंट से जुड़ता है। यह प्रमाणीकरण, अनुरोध स्वरूपण, और प्रतिक्रिया पार्सिंग संभालता है।
2. **एजेंट** – क्लाइंट से `provider.create_agent()` के माध्यम से बनाया गया, एजेंट मॉडल एक्सेस को निर्देशों (सिस्टम प्रॉम्प्ट) और टूल्स के साथ जोड़ता है।
3. **टूल्स** – Python फ़ंक्शन जिन्हें `@tool` से सजाया गया है जिन्हें एजेंट कार्य करने या डेटा प्राप्त करने के लिए कॉल कर सकता है।
4. **सेशन** – एक `AgentSession` ऑब्जेक्ट (जो `agent.create_session()` से बनाया गया है) जो वार्तालाप इतिहास को संग्रहीत करता है, जिससे मल्टी-टर्न संवाद संभव होता है जहाँ एजेंट पूर्व संदर्भ को याद रखता है।

आइए प्रत्येक स्तर को चरण दर चरण बनाएं।


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## @tool डेकोरेटर के साथ टूल्स जोड़ना

टूल एजेंट्स को टेक्स्ट जनरेट करने से परे क्रियाएँ करने देते हैं। `@tool` डेकोरेटर एक सामान्य Python फंक्शन को कुछ ऐसी चीज़ में बदल देता है जिसे एजेंट कॉल कर सकता है।

मुख्य बिंदु:
- मॉडल प्रत्येक पैरामीटर को समझने के लिए `Annotated[type, "description"]` का प्रयोग करें।
- डॉकस्ट्रिंग टूल विवरण बन जाती है जो मॉडल देखता है।
- `approval_mode="never_require"` का मतलब है कि टूल उपयोगकर्ता की पुष्टि के बिना स्वतः चलता है।


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## उपकरणों के साथ एक एजेंट बनाना

अब हम क्लाइंट, निर्देशों, और उपकरणों को एक एजेंट में मिलाते हैं। `instructions` सिस्टम प्रॉम्प्ट के रूप में कार्य करते हैं — वे एजेंट के व्यक्तित्व और व्यवहार को परिभाषित करते हैं।


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## सेशंस के साथ बहु-चरण संवाद

एक `AgentSession` (जो `agent.create_session()` के ज़रिए बनता है) संवाद में सभी संदेशों का ट्रैक रखता है। एक ही सेशन को हर `agent.run()` कॉल में पास करके, एजेंट को पूर्ण संवाद इतिहास तक पहुँच मिलती है और वह पहले के संदेशों का संदर्भ दे सकता है।

हम `tools=[check_destination_availability]` पास करते हैं ताकि एजेंट हर चरण में हमारे उपलब्धता जांचकर्ता को कॉल कर सके।


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## सारांश

इस पाठ में आपने Microsoft Agent Framework के चार स्तंभों का अन्वेषण किया:

| अवधारणा | आपने क्या सीखा |
|---------|------------------|
| **क्लाइंट** | `FoundryChatClient` प्रमाण-आधारित प्रमाणीकरण के साथ Azure OpenAI से जुड़ता है |
| **एजेंट** | `provider.create_agent()` एक मॉडल कनेक्शन को निर्देशों और नाम के साथ बांधता है |
| **टूल्स** | `@tool` डेकोरेटर पायथन फ़ंक्शन्स को एजेंट के कॉल करने के लिए एक्सपोज़ करता है |
| **सेशन** | `agent.create_session()` कई चरणों में वार्तालाप इतिहास बनाए रखता है |

ये निर्माण खंड मिलकर ऐसे एजेंट बनाते हैं जो प्राकृतिक बातचीत कर सकते हैं, बाहरी फ़ंक्शन्स को कॉल कर सकते हैं, और संदर्भ बनाए रख सकते हैं—जो बाद के पाठों में अधिक उन्नत एजेंटिक पैटर्न के लिए आधार है।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
